In [27]:
import pandas as pd
import tensorflow as tf
import sklearn

from tensorflow.keras.layers import Dense, Dropout, Activation, Input
from tensorflow.keras.models import Model
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error

In [4]:
# caminho para o dataset
path = '../archives/games.csv'

df = pd.read_csv(path)
df

,Name,Platform,Year_of_Release,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Score,User_Count,Developer,Rating
0,Wii Sports,Wii,2006.0,Sports,Nintendo,41.36,28.96,3.77,8.45,82.53,76.0,51.0,8,322.0,Nintendo,E
1,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,NaN,NaN,NaN,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.68,12.76,3.79,3.29,35.52,82.0,73.0,8.3,709.0,Nintendo,E
3,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.61,10.93,3.28,2.95,32.77,80.0,73.0,8,192.0,Nintendo,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16714,Samurai Warriors: Sanada Maru,PS3,2016.0,Action,Tecmo Koei,0.00,0.00,0.01,0.00,0.01,NaN,NaN,NaN,NaN,NaN,NaN
16715,LMA Manager 2007,X360,2006.0,Sports,Codemasters,0.00,0.01,0.00,0.00,0.01,NaN,NaN,NaN,NaN,NaN,NaN
16716,Haitaka no Psychedelica,PSV,2016.0,Adventure,Idea Factory,0.00,0.00,0.01,0.00,0.01,NaN,NaN,NaN,NaN,NaN,NaN
16717,Spirits & Spells,GBA,2003.0,Platform,Wanadoo,0.01,0.00,0.00,0.00,0.01,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# removendo os dados que não pretendo utilizar
df = df.drop('Other_Sales', axis=1)
df = df.drop('Global_Sales', axis=1)
df = df.drop('Developer', axis=1)

In [6]:
# quant valores nulos existem em cada coluna
df.isnull().sum()

Name                  2
Platform              0
Year_of_Release     269
Genre                 2
Publisher            54
NA_Sales              0
EU_Sales              0
JP_Sales              0
Critic_Score       8582
Critic_Count       8582
User_Score         6704
User_Count         9129
Rating             6769
dtype: int64

In [7]:
# removendo todas as linhas com valores nulos (não recomendado)
df = df.dropna(axis=0)

In [8]:
# contando os diferentes nomes
df['Name'].value_counts()

Name
Madden NFL 07                                   8
Need for Speed: Most Wanted                     8
LEGO Star Wars II: The Original Trilogy         8
Terraria                                        7
Madden NFL 08                                   7
                                               ..
Brain Age: Train Your Brain in Minutes a Day    1
Wii Fit                                         1
Wii Sports Resort                               1
Mario Kart Wii                                  1
Wii Sports                                      1
Name: count, Length: 4377, dtype: int64

In [9]:
# removendo a coluna 'Nome'
df = df.drop('Name', axis=1)

In [10]:
# vendo os nomes das colunas
df.columns

Index(['Platform', 'Year_of_Release', 'Genre', 'Publisher', 'NA_Sales',
       'EU_Sales', 'JP_Sales', 'Critic_Score', 'Critic_Count', 'User_Score',
       'User_Count', 'Rating'],
      dtype='object')

In [11]:
# separando previsores e resultados
X = df.iloc[:, [0, 1, 2, 3, 7, 8, 9, 10, 11]].values
y_na = df.iloc[:, 4].values
y_eu = df.iloc[:, 5].values
y_jp = df.iloc[:, 6].values

In [12]:
# transformando os valores categoricos
ohe = ColumnTransformer(transformers=[("OneHot", OneHotEncoder(), [0, 2, 3, 8])], remainder='passthrough')
X = ohe.fit_transform(X).toarray()

In [13]:
# vendo o novo formato do X
X.shape

(6825, 303)

In [16]:
# criando as camadas
input_layer = Input(shape=(303,))
dense1 = Dense(units=153, activation='relu')(input_layer)
dense2 = Dense(units=153, activation='relu')(dense1)
output1 = Dense(units=1, activation='linear')(dense2)
output2 = Dense(units=1, activation='linear')(dense2)
output3 = Dense(units=1, activation='linear')(dense2)

In [17]:
# Criando o modelo
regressor = Model(inputs=input_layer, outputs=[output1, output2, output3])

In [ ]:
# compilando
regressor.compile(optimizer='adam', loss='mse')

In [19]:
regressor.fit(X, [y_na, y_eu, y_jp], epochs=500, batch_size=100)

Epoch 1/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - dense_7_loss: 67.4736 - dense_8_loss: 292.5229 - dense_9_loss: 31.0379 - loss: 395.3119  
Epoch 2/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - dense_7_loss: 1.3439 - dense_8_loss: 1.5173 - dense_9_loss: 0.6180 - loss: 3.4905
Epoch 3/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - dense_7_loss: 2.5706 - dense_8_loss: 1.6690 - dense_9_loss: 0.9066 - loss: 5.1807
Epoch 4/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - dense_7_loss: 2.3250 - dense_8_loss: 1.6377 - dense_9_loss: 1.6045 - loss: 5.5564
Epoch 5/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - dense_7_loss: 2.5160 - dense_8_loss: 1.4690 - dense_9_loss: 1.9554 - loss: 5.8620
Epoch 6/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - dense_7_loss: 16.6870 - dense_8_loss: 4.7572 - dense_9_loss: 11.1776 - loss: 32.7939
Epoch 7/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - dense_7_loss: 10.4468 - dense_8_loss: 6.0899 - dense_9_loss: 3.4819 - loss: 20.1716
Epoch 8/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 

In [ ]:
# fazendo as previsões
predict_na, predict_eu, predict_jp = regressor.predict(X)

214/214 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


In [25]:
# valor e media das previsoes (america do norte)
predict_na, predict_na.mean()

(array([[ 1.2958689 ],
        [ 0.8768717 ],
        [ 0.8119038 ],
        ...,
        [-0.03105676],
        [ 0.10179836],
        [ 0.00376785]], dtype=float32),
 np.float32(0.29998234))

In [ ]:
# valor e media das respostas corretas (america do norte)
y_na, y_na.mean()

(array([4.136e+01, 1.568e+01, 1.561e+01, ..., 0.000e+00, 1.000e-02,
        0.000e+00]),
 np.float64(0.3944835164835165))

In [ ]:
# erro minimo (america do norte)
mean_absolute_error(y_na, predict_na)

np.float64(0.3164994832005693)

In [31]:
# valor e media das previsoes (europa)
predict_eu, predict_eu.mean()

(array([[0.8272649 ],
        [0.55934083],
        [0.43711948],
        ...,
        [0.02696794],
        [0.11519828],
        [0.07257345]], dtype=float32),
 np.float32(0.21131226))

In [32]:
# valor e media das respostas corretas (europa)
y_eu, y_eu.mean()

(array([2.896e+01, 1.276e+01, 1.093e+01, ..., 1.000e-02, 0.000e+00,
        1.000e-02]),
 np.float64(0.23608937728937732))

In [33]:
# valor e media das previsoes (japao)
predict_jp, predict_jp.mean()

(array([[0.09317362],
        [0.04907394],
        [0.11873177],
        ...,
        [0.10470387],
        [0.07941507],
        [0.0733261 ]], dtype=float32),
 np.float32(0.09101722))

In [34]:
# valor e media das respostas corretas (japao)
y_jp, y_jp.mean()

(array([3.77, 3.79, 3.28, ..., 0.  , 0.  , 0.  ]),
 np.float64(0.06415824175824175))

In [35]:
# erro minimo (europa)
mean_absolute_error(y_eu, predict_eu)

np.float64(0.23805780701707133)

In [36]:
# erro minimo (japao)
mean_absolute_error(y_jp, predict_jp)

np.float64(0.12123752737058388)